# PhenoCycler cell identities and protein expression

Use this notebook for pancreas or pancreatic lymph node (pLN). The first plots use complete sections. Select a donor to compare a section with its disease group, open the full donor UMAP, and explore the native spatial measurements in Scanpy. Cell identities and review categories have separate plots.

In [ ]:
from pathlib import Path
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from IPython.display import display, Markdown
here = Path.cwd()
release = os.environ.get('PHENOCYCLER_DYNAMIC_RELEASE')
name = 'phenocycler_gate_calibration_20261116'
candidates = [Path(release)] if release else [
    p / sub / name for p in (here, *here.parents)
    for sub in ('releases', 'analysis/phenocycler/releases')
]
root = next((p for p in candidates if (p / 'scanpy' / 'release.json').is_file()), None)
if root is None:
    raise FileNotFoundError('Set PHENOCYCLER_DYNAMIC_RELEASE to the private fitted release containing scanpy/release.json.')
modules = root / 'scanpy'
sys.path.insert(0, str(modules))
import release_data as rd
import plots as vis
import donor_map_plots as donor_maps

TISSUE = 'pancreas'            # pancreas or pLN
DONOR = '6593'                  # choose any donor listed below
DISEASE = None                 # None selects all recorded disease states
PANEL = 'shared'               # exact availability groups appear below
COHORT_DISPLAY_PER_DONOR = 10000
sc.settings.verbosity = 0
label_palette = donor_maps.label_palette(TISSUE)
def show(fig):
    display(fig)
    plt.close(fig)
sections = pd.read_csv(rd.BASE / 'section_manifest.csv', dtype={'donor': str})
sections = sections[sections.tissue.eq(rd.key(TISSUE))]
group = sections if DISEASE is None else sections[sections.disease.eq(DISEASE)]
donor = rd.normalize_donor(TISSUE, DONOR)
display(group.groupby('disease').agg(donors=('donor','nunique'), eligible_cells=('eligible_cells','sum')))
display(group[['donor','disease','age_years','sex','eligible_cells','markers']].sort_values('donor').reset_index(drop=True))

## Proteins measured in the same donors

A shared protein has a finite measured intensity in every selected donor section. Each other panel lists proteins measured in the same subset of donors. A marker can have measured intensity and an unidentifiable positivity gate. Its intensity remains available, and the positive fraction uses only cells with an assessable fitted decision.

In [ ]:
panels = vis.coverage_groups(TISSUE, disease=DISEASE)
display(pd.DataFrame([{'panel': k, 'donors': ', '.join(v['donors']), 'proteins': ', '.join(v['markers'])} for k, v in panels.items()]))
assert PANEL in panels, list(panels)

## Grouped protein profiles

Each donor contributes its own mean log(1 + aligned signal) and positive fraction for the displayed cell type. Dot color scales the means within each marker column. Dot area shows the mean fraction positive among assessable cells. Each heatmap independently scales each nonconstant marker column from its minimum to its maximum across the labels displayed. Compare absolute values between identities and QC using the numeric profiles returned by `vis.profiles`. Grey indicates missing signal or a constant marker. A label with maximal signal in every measured column receives a quantitative profile.

In [ ]:
for qc in (False, True):
    for kind, fig in vis.marker_figures(TISSUE, disease=DISEASE, panel=PANEL, qc=qc).items():
        show(fig)

## Per-donor protein profiles

The selected donor uses every available marker. Its means and fractions use its own complete eligible population.

In [ ]:
for qc in (False, True):
    for kind, fig in vis.marker_figures(TISSUE, donor=donor, qc=qc).items():
        show(fig)

## Cell type frequencies

The denominator is the number of eligible cells in the section, including cells with labels held for review. Bars show the donor median within each disease group. Points show individual donors. Rare frequencies have separate panels. The individual-donor figures use the same whole-section denominator.

In [ ]:
for selection in ({'disease': DISEASE}, {'donor': donor}):
    display(Markdown('### '+('Donor '+donor if 'donor' in selection else 'Disease groups')))
    for qc in (False, True):
        for fig in vis.frequency_figures(TISSUE, qc=qc, **selection).values():
            show(fig)

## Cohort UMAP by protein expression

The cohort embedding uses a deterministic sample based on a hash of each native object ID. Each donor contributes up to the chosen display limit. Rows missing any protein in this embedding are omitted. The table reports admitted, drawn and plotted cells for every section. The expression profiles and frequencies above use the full sections. UMAP coordinates serve as a visual check and do not determine cell labels.

In [ ]:
def show_embedding(obj):
    print(f"{obj.n_vars} common measured input proteins:")
    print(', '.join(obj.uns['display_markers']))
    report = obj.uns['display_accounting'].copy()
    report['plotted_percent_of_admitted'] = (report.fully_measured_plotted / report.admitted * 100).round(3)
    display(report)
    for qc, title in [(False, 'Cell identities'), (True, 'Assignment QC')]:
        part = obj[obj.obs.qc.eq(qc)].copy()
        if len(part):
            with plt.rc_context({'figure.figsize': (8,5)}):
                ax = sc.pl.umap(part, color='label', title=title, palette=label_palette, legend_fontsize=6, size=5, show=False, frameon=False)
                show(ax.figure)
cohort = vis.embedding(TISSUE, group.donor.tolist(), per_donor=COHORT_DISPLAY_PER_DONOR)
show_embedding(cohort)

## Full per-donor UMAPs

Each section has its own UMAP fitted with every eligible cell that has a finite intensity for every protein measured in that section. Open the density, identity and QC figures for any donor below. Each density hexagon counts all complete cells at that location, including identity and QC categories. Change `DONOR` and rerun to work with that donor’s cells and coordinates in Scanpy. Axes can differ between independently fitted maps. The tissue and CD68–CD163 scatterplots use the same complete profiles as the selected donor’s UMAP.

In [ ]:
display(Markdown(donor_maps.gallery(TISSUE, link_directory=Path(os.path.relpath(rd.BASE, here)))))
single = donor_maps.load_map(TISSUE, donor)
fit = single.uns['donor_umap']
print(f"Donor {donor} · {fit['eligible_cells']:,} eligible · {fit['incomplete_cells']:,} incomplete · "
      f"{single.n_obs:,} fitted and plotted")
print(f"{len(fit['markers'])} input proteins: " + ', '.join(fit['markers']))
show(donor_maps.density_figure(single))
for basis in ('umap', 'spatial', 'protein_pair'):
    for qc in (False, True):
        show(donor_maps.figure(single, basis=basis, qc=qc, palette=label_palette))


## Work with the measurements in Scanpy

The loader returns an AnnData for any donor. Omit `n_cells` for every eligible object. The raw aligned intensity is in `X`, fitted marker decisions are in `layers['fitted_state']`, and centroids are in `obsm['spatial']`. Positive, negative and unassessed decisions are encoded as 1, 0 and −1. The observation table contains the native object ID, label, disease and cell-specific uncertainty, including the pancreas refinement and pLN epithelial-veto information. The plot section can be relocated as a directory.

`single.obsm["X_umap"]` contains the verified full donor coordinates. Every protein in the donor fit is available directly. For example, use `sc.pl.umap(single, color="INS")` for a pancreas insulin overlay or `sc.pl.umap(single, color="CD163")` for either tissue.

In [ ]:
example = rd.load_donor(TISSUE, donor, markers=['CD68','CD163'], n_cells=1000)
display(example)
display(example.var)
print(', '.join(example.obs.columns))
# For an entire section with the full measured panel:
# entire_section = rd.load_donor(TISSUE, donor)

## Per-donor islet trajectories

Open the [per-donor trajectory notebook](07_islet_trajectories.ipynb) to select the features that define pseudotime, plot other features along it, fit excess-zero count models, or run the Seurat scSLIDE protein trajectory. The panel below shows measured core and rim inflammation for this donor.

In [ ]:
if rd.key(TISSUE) == 'pancreas':
    import json
    from matplotlib.collections import LineCollection
    location = rd.ROOT / 'islet_trajectories' / ('donor_id=' + donor)
    if location.exists():
        islets = pd.read_parquet(location / 'islets.parquet')
        display(islets.groupby('assessed').size().rename('islets').to_frame())
        view = islets[islets.assessed].set_index('islet_id')
        dims = ['core_immune_per_mm2', 'peri_immune_per_mm2']
        coordinates = view[dims].apply(np.log1p)
        edges = json.loads((location / 'tree.json').read_text())
        pairs = [(e['source_islet'],e['target_islet']) for e in edges if e['source_islet'] in view.index and e['target_islet'] in view.index]
        fig, ax = plt.subplots(figsize=(7,5), layout='constrained')
        if pairs:
            lines = coordinates.loc[[key for edge in pairs for key in edge]].to_numpy().reshape(-1,2,2)
            ax.add_collection(LineCollection(lines, colors='#666666', linewidths=.7, alpha=.6))
        points = ax.scatter(coordinates.iloc[:,0],coordinates.iloc[:,1],c=np.log1p(view.combined_immune_per_mm2),cmap='viridis',s=12)
        root = coordinates[view.tree_root]
        if len(root):
            ax.scatter(root.iloc[:,0],root.iloc[:,1],s=90,facecolors='none',edgecolors='#272727',linewidth=1.4,label='Root')
            ax.legend()
        fig.colorbar(points,ax=ax,label='log(1 + combined immune cells/mm²)')
        ax.set(xlabel='log(1 + core immune cells/mm²)',ylabel='log(1 + surrounding immune cells/mm²)',title=f'Donor {donor} · islet inflammation')
        show(fig)
    else:
        print('The companion islet_trajectories directory is required for this pancreatic plot.')
